# Day 02 — Boolean Indexing + Filtering
**Estimated time:** 45–60 min  
**Dataset:** `sales_orders.csv`

## Learning Objectives
- Master multi-condition boolean indexing with `.loc`, `.iloc`, and `.query()`
- Understand the performance and readability trade-offs between filter approaches
- Build filters that mirror SAP transaction logic (VA05, VF05)


In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "../datasets"
df = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")

# Fix types
df["ERDAT"] = pd.to_datetime(df["ERDAT"], errors="coerce")
df["NETWR"] = pd.to_numeric(df["NETWR"], errors="coerce")
df["MENGE"] = pd.to_numeric(df["MENGE"], errors="coerce")

print("Shape:", df.shape)
print(df.dtypes)
display(df.head())


In [ ]:
# ── 1. Single-condition filter: open orders only ─────────────────────────────
open_orders = df[df["STATUS"] == "OPEN"]
print(f"Open orders: {len(open_orders):,} of {len(df):,} total")


In [ ]:
# ── 2. Open orders older than 90 days ────────────────────────────────────────
cutoff_90 = pd.Timestamp.today() - pd.Timedelta(days=90)

# Bracket syntax
stale_bracket = df[(df["STATUS"] == "OPEN") & (df["ERDAT"] < cutoff_90)]

# .query() syntax — cleaner for complex conditions
stale_query = df.query("STATUS == 'OPEN' and ERDAT < @cutoff_90")

print(f"Stale open orders (bracket): {len(stale_bracket):,}")
print(f"Stale open orders (.query):  {len(stale_query):,}")
assert len(stale_bracket) == len(stale_query), "Mismatch — check your logic"


In [ ]:
# ── 3. High-value orders in a specific sales org ─────────────────────────────
# Change VKORG value to match your dataset
target_vkorg = df["VKORG"].value_counts().index[0]   # biggest org for demo

high_value = df.query("NETWR > 10_000 and VKORG == @target_vkorg")
print(f"Orders >$10k in VKORG {target_vkorg}: {len(high_value):,}")
print(high_value[["VBELN","POSNR","KUNNR","NETWR","STATUS"]].sort_values("NETWR", ascending=False).head(10))


In [ ]:
# ── 4. Multi-condition filter: OPEN + high-value + specific channel ───────────
# Demonstrates chaining conditions cleanly with .query()
result = df.query(
    "STATUS == 'OPEN' "
    "and NETWR > 5_000 "
    "and ERDAT >= '2024-01-01'"
)
print(f"Qualifying orders: {len(result):,}")
print(result.groupby("VKORG")["NETWR"].agg(["count","sum"]).sort_values("sum", ascending=False))


In [ ]:
# ── 5. .loc vs .iloc — understand the difference cold ────────────────────────
# .loc  → label-based (column names, row index labels)
# .iloc → integer position-based (0-indexed positions)

print("── .loc: first 3 rows, specific columns ──")
print(df.loc[0:2, ["VBELN", "KUNNR", "NETWR"]])      # inclusive on both ends

print("\n── .iloc: first 3 rows, first 4 columns ──")
print(df.iloc[0:3, 0:4])                               # exclusive on right end

# Gotcha: .loc with a string index vs .iloc
# When index is default RangeIndex, df.loc[2] == df.iloc[2]
# After filtering, index labels are PRESERVED — .iloc[0] != .loc[0] for sliced df
filtered = df[df["STATUS"] == "OPEN"].copy()
print(f"\nfiltered.index[0] = {filtered.index[0]}")     # not necessarily 0
print(f"filtered.iloc[0]['VBELN'] = {filtered.iloc[0]['VBELN']}")   # first record
print(f"filtered.loc[filtered.index[0]]['VBELN'] = {filtered.loc[filtered.index[0]]['VBELN']}")


In [ ]:
# ── 6. isin() for set membership ─────────────────────────────────────────────
# Equivalent to SAP WHERE SPART IN ('01','02','03')
target_divisions = df["SPART"].dropna().unique()[:3].tolist()
print(f"Filtering for SPART in: {target_divisions}")

div_filter = df[df["SPART"].isin(target_divisions)]
print(f"Records: {len(div_filter):,}")


In [ ]:
# ── 7. String contains — flexible text matching ──────────────────────────────
# Find customers starting with a specific prefix (useful when IDs are structured)
prefix = df["KUNNR"].astype(str).str[:2].value_counts().index[0]
cust_filter = df[df["KUNNR"].astype(str).str.startswith(prefix)]
print(f"Orders for customers starting with '{prefix}': {len(cust_filter):,}")


---
## Daily Prompt

**How many OPEN orders have not been updated (ERDAT) in over 60 days? Segment by VKORG (sales organization) and sort by count descending.**

```python
# YOUR SOLUTION
cutoff_60 = pd.Timestamp.today() - pd.Timedelta(days=60)

result = (
    df.query("STATUS == 'OPEN' and ERDAT < @cutoff_60")
    # YOUR SOLUTION: group by VKORG, count, sort
)
print(result)
```

> **Hint:** After filtering, use `.groupby("VKORG").size().sort_values(ascending=False)`.  
> Then ask: which sales orgs have the highest backlog risk?  
> Cross-reference with total order value per org using `.agg({"VBELN": "count", "NETWR": "sum"})`.
